In [14]:
import pandas as pd
import plotly.express as px
from pathlib import Path

# ============================
# 1) Load hotels CSV
# ============================

BASE_DIR = Path().resolve()  # notebook in 04-Maps
csv_path_hotels = BASE_DIR.parent / "03-booking_scraper" / "hotels.csv"
df_hotels = pd.read_csv(csv_path_hotels)

# ============================
# 2) Prepare data
# ============================

numeric_cols = ["review_score", "lat", "lon"]
for col in numeric_cols:
    df_hotels[col] = pd.to_numeric(df_hotels[col], errors="coerce")

# Keep only valid rows
df_valid = df_hotels.dropna(subset=["review_score", "lat", "lon"]).copy()

if df_valid.empty:
    print("Pas d'hôtels valides.")
else:
    # Top 20 hotels PER site (site_id)
    df_hotels_top = (
        df_valid
        .sort_values("review_score", ascending=False)
        .groupby("site_id", group_keys=False)
        .head(20)
        .copy()
    )

    # Marker size from review_score
    min_score = df_hotels_top["review_score"].min()
    max_score = df_hotels_top["review_score"].max()
    df_hotels_top["size_marker"] = 8 + (df_hotels_top["review_score"] - min_score) * (
        12 / max(1e-6, (max_score - min_score))
    )

    # ============================
    # 3) Hotels map over France
    # ============================

    fig_hotels = px.scatter_map(
        df_hotels_top,
        lat="lat",
        lon="lon",
        size="size_marker",
        color="review_score",
        color_continuous_scale=[(0, "blue"), (0.5, "purple"), (1, "red")],
        hover_name="name",
        hover_data={
            "destination": True,
            "site_id": True,
            "address": True,
            "review_score": True,
            "lat": False,
            "lon": False,
        },
        map_style="open-street-map",
    )

    fig_hotels.update_layout(
        map=dict(
            center={"lat": 46.6, "lon": 2.4},  # centre France
            zoom=4.8,
        ),
        height=750,
        width=750,
        margin=dict(r=0, l=0, t=40, b=0),
        title="Top 20 hôtels par site (review_score)",
    )

    fig_hotels.show()
